# Spot matching

For each fish and side, matches every day-1 spot to the nearest day-2 spot (distance between bounding-box centres) and rewrites the day-2 label file in `data/<GROUP>/labels/spots/` in place, so that line *i* is the same spot on both days. Set `GROUP` to `stressed` or `misc`.

In [1]:
import os
import time
import numpy as np
from PIL import Image

In [3]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "fish_ids.csv").is_file())
GROUP = "misc"
DATA = ROOT / "data" / GROUP

image_dir = DATA / "images" / "normalized_operculums"
labels_dir = DATA / "labels" / "spots"


def load_paths(directory):
    """
    Load all files from a directory into a dictionary.

    Dictionary format:
        {
            filename_without_extension: full_file_path
        }
    """
    paths = {}

    for file_name in sorted(os.listdir(directory)):
        key = os.path.splitext(file_name)[0]
        paths[key] = os.path.join(directory, file_name)

    return paths


img_paths = load_paths(image_dir)
lbl_paths = load_paths(labels_dir)


def load_annotations(label_paths, remove_class=False):
    """
    Load annotation files.

    Args:
        label_paths (dict): Dictionary of label file paths.
        remove_class (bool): 
            If True, removes the first value from each line.
            This is useful when only polygon coordinates are needed.

    Returns:
        dict:
            {
                filename: [
                    [annotation values],
                    [annotation values],
                    ...
                ]
            }
    """
    annotations = {}

    for key, path in label_paths.items():
        with open(path, "r") as file:
            lines = file.readlines()

        if remove_class:
            annotations[key] = [
                line.split()[1:] for line in lines
            ]
        else:
            annotations[key] = [
                line.split() for line in lines
            ]

    return annotations


lbls_det = load_annotations(lbl_paths, remove_class=True)

lbls_det_with_class = load_annotations(lbl_paths, remove_class=False)


def sorting_custom(file_key):
    """
    Sort files based on fish ID and side.

    Expected filename structure:
        something_<fish_id>_..._<side>_something

    Example:
        fish_10_S_day1_id-51_L_h
    """
    try:
        parts = file_key.split("_")

        fish_id = int(parts[1])
        side = parts[-2]

        return fish_id, side

    except Exception as error:
        print(f"Could not sort file key: {file_key}. Error: {error}")
        return float("inf"), ""


sorted_fish_paths = dict(
    sorted(img_paths.items(), key=lambda x: sorting_custom(x[0]))
)

fnames = list(sorted_fish_paths.keys())


def polygon_to_bounding_box(polygon):
    """
    Convert polygon coordinates to a bounding box.

    Args:
        polygon (np.ndarray): Array of shape (N, 2), where each row is (x, y).

    Returns:
        tuple:
            (min_x, min_y, max_x, max_y)
    """
    x_values = polygon[:, 0]
    y_values = polygon[:, 1]

    min_x = int(np.min(x_values))
    max_x = int(np.max(x_values))
    min_y = int(np.min(y_values))
    max_y = int(np.max(y_values))

    return min_x, min_y, max_x, max_y


def euclidean_distance(x1, y1, x2, y2):
    """
    Calculate Euclidean distance between two points.
    """
    return np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


def calculate_annotation_centers(fnames, image_paths, annotations):
    """
    Calculate center points for each polygon annotation.

    For each annotation:
    - Convert normalized polygon coordinates to pixel coordinates.
    - Convert polygon to bounding box.
    - Calculate the center of the bounding box.

    Returns:
        dict:
            {
                filename: {
                    annotation_index: (center_x, center_y)
                }
            }
    """
    centers_global = {}

    for fn in fnames:
        img = Image.open(image_paths[fn])
        img_width, img_height = img.size

        centers = {}

        for annotation_index, annotation in enumerate(annotations[fn]):
            vertices = [float(x) for x in annotation]

            pixel_vertices = np.array([
                int(vertices[i] * img_width) if i % 2 == 0
                else int(vertices[i] * img_height)
                for i in range(len(vertices))
            ]).reshape(-1, 2)

            x_min, y_min, x_max, y_max = polygon_to_bounding_box(pixel_vertices)

            center_x = int((x_max - x_min) / 2) + x_min
            center_y = int((y_max - y_min) / 2) + y_min

            centers[annotation_index] = (center_x, center_y)

        centers_global[fn] = centers

    return centers_global


cords_global = calculate_annotation_centers(
    fnames=fnames,
    image_paths=img_paths,
    annotations=lbls_det
)


start_time = time.time()

for pre_key, post_key in np.array(list(cords_global.keys())).reshape(-1, 2):

    matched_annotations = []

    for pre_spot_index, (pre_x, pre_y) in cords_global[pre_key].items():

        distances = {}

        for post_spot_index, (post_x, post_y) in cords_global[post_key].items():

            distance = euclidean_distance(
                pre_x,
                pre_y,
                post_x,
                post_y
            )

            print(
                f"{pre_key}:{post_key}: "
                f"Spot {pre_spot_index} -> Spot {post_spot_index} = {distance}"
            )

            distances[f"{pre_spot_index}_{post_spot_index}"] = float(distance)

        distance_keys = np.array(list(distances.keys()))
        distance_values = np.array(list(distances.values()))

        min_index = np.argmin(distance_values)

        matched_pair = distance_keys[min_index]
        matched_distance = distance_values[min_index]

        pre_match, post_match = matched_pair.split("_")

        print(
            f"Minimum distance between pre-stress spot {pre_match} "
            f"and post-stress spot {post_match} is {matched_distance}"
        )

        matched_annotations.append([
            float(x) for x in lbls_det_with_class[post_key][min_index]
        ])

    output_label_path = os.path.join(labels_dir, f"{post_key}.txt")

    with open(output_label_path, "w") as file:
        for annotation in matched_annotations:
            annotation_line = " ".join(str(value) for value in annotation)
            file.write(f"{annotation_line}\n")

    print(f"{post_key} - Done")


print(f"--- {time.time() - start_time:.2f} seconds ---")

fish_31_S_day1_id-11_L_h:fish_31_S_day2_id-222_L_h: Spot 0 -> Spot 0 = 583.6719969297825
fish_31_S_day1_id-11_L_h:fish_31_S_day2_id-222_L_h: Spot 0 -> Spot 1 = 231.39792566053828
fish_31_S_day1_id-11_L_h:fish_31_S_day2_id-222_L_h: Spot 0 -> Spot 2 = 25.495097567963924
fish_31_S_day1_id-11_L_h:fish_31_S_day2_id-222_L_h: Spot 0 -> Spot 3 = 136.69308687713507
fish_31_S_day1_id-11_L_h:fish_31_S_day2_id-222_L_h: Spot 0 -> Spot 4 = 168.86681142249356
fish_31_S_day1_id-11_L_h:fish_31_S_day2_id-222_L_h: Spot 0 -> Spot 5 = 582.2860121967555
fish_31_S_day1_id-11_L_h:fish_31_S_day2_id-222_L_h: Spot 0 -> Spot 6 = 168.14577009250038
fish_31_S_day1_id-11_L_h:fish_31_S_day2_id-222_L_h: Spot 0 -> Spot 7 = 80.60397012554654
fish_31_S_day1_id-11_L_h:fish_31_S_day2_id-222_L_h: Spot 0 -> Spot 8 = 247.13154391942766
fish_31_S_day1_id-11_L_h:fish_31_S_day2_id-222_L_h: Spot 0 -> Spot 9 = 635.6138765005056
fish_31_S_day1_id-11_L_h:fish_31_S_day2_id-222_L_h: Spot 0 -> Spot 10 = 677.6031286822692
fish_31_S_day1